# 02 · Train LoRA on Nemotron-3-Nano-30B (Kaggle GPU)

Upload this notebook to Kaggle, attach the competition dataset and the
`metric/nemotron-3-nano-30b-a3b-bf16` model, then run top to bottom.

Output: `/kaggle/working/submission.zip` containing the LoRA adapter.

## 0. Environment

Kaggle's notebook images include PyTorch + Transformers; we add PEFT/TRL/datasets
plus the Mamba kernels Nemotron requires and bitsandbytes so the 30B base fits
alongside training state on a T4 x2 (32 GB) or L4 x4 (96 GB) instance.

**Required GPU**: select **GPU T4 x2** or **GPU L4 x4** in the kernel settings.
The free P100 (sm_60) is not supported by current PyTorch wheels.

In [ ]:
%pip install -q -U peft accelerate trl datasets bitsandbytes
%pip install -q --no-build-isolation mamba-ssm causal-conv1d

## 1. Load the prebuilt SFT dataset

Expects a Kaggle Dataset attached to the notebook containing `sft_v1.parquet`
(produced locally via `scripts/build_sft_data.py`). The mount point varies by
platform/CLI version, so we glob a few candidates instead of hard-coding one.

In [ ]:
import glob, json, os
import polars as pl
from datasets import Dataset

_candidates = sorted(set(
    glob.glob('/kaggle/input/wonderland-sft-v1/sft_v1.parquet')
    + glob.glob('/kaggle/input/datasets/*/wonderland-sft-v1/sft_v1.parquet')
    + glob.glob('/kaggle/input/*/sft_v1.parquet')
))
assert _candidates, 'sft_v1.parquet not found under /kaggle/input — attach the wonderland-sft-v1 dataset'
SFT_PATH = _candidates[0]
print('using SFT data at', SFT_PATH)

df = pl.read_parquet(SFT_PATH)
print('total rows:', df.height)
print(df.group_by(['category', 'source']).agg(pl.len().alias('n')).sort(['category', 'source']))

## 2. Stratified 90/10 train/val split

Hold out ~10% of every category for offline scoring after training so we can
spot-check the adapter before submitting. The split is deterministic
(`seed=0`) and stratified — each category contributes ~10% of its rows to val.

In [ ]:
SPLIT_SEED = 0
VAL_FRACTION = 0.10

df = df.with_row_index('_row')
val_parts, train_parts = [], []
for cat in sorted(df['category'].unique().to_list()):
    sub = df.filter(pl.col('category') == cat).sample(
        fraction=1.0, shuffle=True, seed=SPLIT_SEED
    )
    n_val = max(1, int(round(sub.height * VAL_FRACTION)))
    val_parts.append(sub.head(n_val))
    train_parts.append(sub.tail(sub.height - n_val))

df_val = pl.concat(val_parts).sort('_row').drop('_row')
df_train = pl.concat(train_parts).sort('_row').drop('_row')
df = df.drop('_row')

print(f'train rows: {df_train.height}')
print(f'val rows:   {df_val.height}')
print('val per-category:')
print(df_val.group_by('category').agg(pl.len().alias('n')).sort('category'))

## 3. Build the HuggingFace Dataset (messages format)

In [ ]:
rows = [{'messages': json.loads(m)} for m in df_train['messages'].to_list()]
ds = Dataset.from_list(rows)
print(ds)
print('example messages:')
for m in ds[0]['messages']:
    print(f"[{m['role']}]", m['content'][:200])

## 4. Load Nemotron-3-Nano-30B + attach LoRA (QLoRA: 4-bit base)

Nemotron-3-Nano-30B-A3B is ~63 GB in BF16 — too large for T4 x2 (32 GB) directly.
We load the base in 4-bit NF4 via bitsandbytes (≈ 15 GB), call
`prepare_model_for_kbit_training`, and attach a rank-32 LoRA on the Mamba
(`in_proj`, `out_proj`) and MLP (`up_proj`, `down_proj`) projections. The grader
merges the same adapter onto the BF16 base at eval time, so QLoRA training is
compatible with the BF16 evaluation path.

In [ ]:
import site, subprocess
import torch

# Diagnostics: surface the allocated accelerator BEFORE we try to load a 30B model.
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
print(f'torch.cuda.device_count(): {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {torch.cuda.get_device_name(i)}  '
          f'{props.total_memory / 1024**3:.1f} GiB  sm_{props.major}{props.minor}')
print()
if torch.cuda.device_count() < 2:
    raise RuntimeError(
        f'Got {torch.cuda.device_count()} GPU(s); the 30B model needs T4 x2 (32 GiB) or larger. '
        'Open this kernel in the Kaggle editor, Settings -> Accelerator -> select GPU T4 x2 (not GPU T4), '
        'then Save Version -> Save & Run All.'
    )
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

# Some Nemotron builds need this CUTLASS DSL helper (shipped from a Kaggle utility script).
cutlass_pkg_path = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/'
if os.path.exists(cutlass_pkg_path):
    site.addsitedir(cutlass_pkg_path)

import kagglehub
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
OUTPUT_DIR = '/kaggle/working'
LORA_RANK = 32  # grader enforces max_lora_rank=32

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Help the allocator deal with fragmentation when sharding across two small GPUs.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Default `device_map='auto'` packs everything onto GPU 0 first; on T4 x2 that
# means GPU 0 OOMs while GPU 1 sits empty. Set explicit per-device budgets so
# accelerate actually splits the model.
n_gpus = torch.cuda.device_count()
per_gpu_budget_gib = 13  # leave ~2 GiB headroom on each 15 GiB T4 for activations + optimiser
max_memory = {i: f'{per_gpu_budget_gib}GiB' for i in range(n_gpus)}
max_memory['cpu'] = '24GiB'  # last-resort overflow

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb,
    device_map='auto',
    max_memory=max_memory,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
print('device map after load:')
if hasattr(model, 'hf_device_map'):
    for k, v in sorted(model.hf_device_map.items())[:20]:
        print(f'  {k:60s} -> {v}')
    print(f'  (total entries: {len(model.hf_device_map)})')
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r'.*\\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. SFT training

Single-epoch SFT with TRL. Tune batch size / grad-accum / LR to fit your GPU.
On L4 x4 the values below are a reasonable starting point; on T4 x2 drop
`per_device_train_batch_size` to 1 and add 4-bit base loading.

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=25,
    save_strategy='no',
    max_seq_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds,
    tokenizer=tokenizer,
)
trainer.train()

## 6. Held-out validation

Run inference on the held-out split and score it locally with the same rule
the grader uses (last `\boxed{...}` → fallback to last number → exact-match or
rel-tol ≤ 1e-2). Edit `VAL_CAP` to trade off wall time vs. signal — we cap to
~300 rows by default sampled uniformly across categories.

Generation params mirror the grader: `do_sample=False`, `temperature=0.0`,
`max_new_tokens=7680`. We prefer vLLM with the trained LoRA loaded as a
request-time adapter; if vLLM can't initialise in the Kaggle env (e.g. no
wheel for the Mamba kernels), we fall back to `model.generate` in small
batches with the same params.

### 6a. Validation config + prompt construction

In [ ]:
VAL_CAP = 300  # set to None to score the entire val split (slow!)
VAL_BATCH_SIZE = 4  # only used by the transformers fallback

# Save the LoRA adapter to disk first so vLLM can load it from a path.
ADAPTER_DIR = OUTPUT_DIR + '/adapter_for_val'
model.save_pretrained(ADAPTER_DIR)
print('saved adapter to', ADAPTER_DIR)

# Uniformly cap val across categories (preserves stratification).
if VAL_CAP is not None and df_val.height > VAL_CAP:
    cats = sorted(df_val['category'].unique().to_list())
    per_cat = max(1, VAL_CAP // len(cats))
    capped = []
    for cat in cats:
        sub = df_val.filter(pl.col('category') == cat).sample(
            fraction=1.0, shuffle=True, seed=SPLIT_SEED
        )
        capped.append(sub.head(per_cat))
    df_val_eval = pl.concat(capped)
else:
    df_val_eval = df_val

print(f'scoring {df_val_eval.height} val rows')
print(df_val_eval.group_by('category').agg(pl.len().alias('n')).sort('category'))

# For inference the assistant turn is what the model must produce; use only
# [system, user] and let chat-template add the assistant-start marker.
val_records = []
for row in df_val_eval.iter_rows(named=True):
    msgs = json.loads(row['messages'])
    inference_msgs = [m for m in msgs if m['role'] != 'assistant']
    prompt_text = tokenizer.apply_chat_template(
        inference_msgs, tokenize=False, add_generation_prompt=True
    )
    val_records.append({
        'id': row['id'],
        'category': row['category'],
        'prompt_text': prompt_text,
        'target': row['answer'],
    })

print(f'built {len(val_records)} prompts; first prompt tail:')
print('...' + val_records[0]['prompt_text'][-300:])

### 6b. Local metric (mirrors the grader)

Re-implemented inline so this notebook stays self-contained — the Kaggle
runtime doesn't have our repo.

In [ ]:
import re

_BOXED_RE = re.compile(r'\\boxed\{(.*?)\}', re.DOTALL)
_NUM_RE = re.compile(r'-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?')
REL_TOL = 1e-2

def extract_answer(text: str) -> str:
    boxed = _BOXED_RE.findall(text)
    if boxed:
        return boxed[-1].strip()
    nums = _NUM_RE.findall(text)
    if nums:
        return nums[-1]
    return text.strip()

def _try_float(s: str):
    try:
        return float(s)
    except (ValueError, TypeError):
        return None

def is_correct(prediction: str, target: str) -> bool:
    pred = extract_answer(prediction).strip()
    tgt = target.strip()
    if pred == tgt:
        return True
    p_num, t_num = _try_float(pred), _try_float(tgt)
    if p_num is not None and t_num is not None:
        if t_num == 0.0:
            return abs(p_num) <= REL_TOL
        return abs(p_num - t_num) / abs(t_num) <= REL_TOL
    return False

# Quick self-test so a regression in the regex shows up immediately.
assert extract_answer('the answer is \\boxed{42}') == '42'
assert is_correct('\\boxed{42.005}', '42')  # within rel-tol
assert not is_correct('\\boxed{nope}', 'yes')
assert is_correct('I get 7 then 13', '13')  # last-number fallback
print('metric self-test ok')

### 6c. Run inference (vLLM preferred, transformers fallback)

vLLM matches the grader exactly. If it can't be imported / initialised in this
environment we drop to plain `model.generate` with the same params — slower but
always available.

In [ ]:
predictions = [None] * len(val_records)
inference_backend = None

try:
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    print('attempting vLLM inference...')
    # Free GPU memory held by the training model + optimiser before booting vLLM.
    import gc
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

    llm = LLM(
        model=MODEL_PATH,
        dtype='bfloat16',
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=LORA_RANK,
        max_model_len=8192,
        gpu_memory_utilization=0.85,
    )
    sampling = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=7680)
    lora_req = LoRARequest('trained_adapter', 1, ADAPTER_DIR)

    prompts = [r['prompt_text'] for r in val_records]
    outputs = llm.generate(prompts, sampling, lora_request=lora_req)
    # `llm.generate` returns outputs in the same order as the input prompts.
    for i, o in enumerate(outputs):
        predictions[i] = o.outputs[0].text
    inference_backend = 'vllm'
    print('vLLM inference done')
except Exception as e:
    print(f'vLLM unavailable ({type(e).__name__}: {e}); falling back to transformers.generate')
    inference_backend = 'transformers'

if inference_backend == 'transformers':
    model.eval()
    for start in range(0, len(val_records), VAL_BATCH_SIZE):
        batch = val_records[start:start + VAL_BATCH_SIZE]
        enc = tokenizer(
            [r['prompt_text'] for r in batch],
            return_tensors='pt', padding=True, truncation=False,
        ).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                **enc,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                max_new_tokens=7680,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        for i, ids in enumerate(out_ids):
            prompt_len = enc['input_ids'][i].shape[0]
            gen_ids = ids[prompt_len:]
            predictions[start + i] = tokenizer.decode(gen_ids, skip_special_tokens=True)
        print(f'  generated rows {start}..{start + len(batch) - 1}')

print(f'inference backend: {inference_backend}')
print(f'predictions filled: {sum(p is not None for p in predictions)} / {len(predictions)}')

### 6d. Per-category accuracy

In [ ]:
scored = []
for rec, pred in zip(val_records, predictions):
    pred = pred or ''
    scored.append({
        'id': rec['id'],
        'category': rec['category'],
        'target': rec['target'],
        'prediction': pred,
        'extracted': extract_answer(pred),
        'correct': is_correct(pred, rec['target']),
    })

scored_df = pl.DataFrame(scored)
overall_acc = scored_df['correct'].mean()

per_cat = scored_df.group_by('category').agg([
    pl.len().alias('n'),
    pl.col('correct').mean().alias('accuracy'),
]).sort('category')

print(f'overall val accuracy: {overall_acc:.4f} ({scored_df["correct"].sum()}/{scored_df.height})')
print(per_cat)

### 6e. Debug printouts — one example per category

In [ ]:
for cat in sorted(scored_df['category'].unique().to_list()):
    ex = scored_df.filter(pl.col('category') == cat).head(1).to_dicts()[0]
    rec = next(r for r in val_records if r['id'] == ex['id'])
    print('=' * 80)
    print(f"CATEGORY: {cat}   id={ex['id']}   correct={ex['correct']}")
    print('=' * 80)
    print('PROMPT (last 200 chars):')
    print('...' + rec['prompt_text'][-200:])
    print()
    print('GENERATED:')
    print(ex['prediction'])
    print()
    print(f"TRUE ANSWER:  {ex['target']}")
    print(f"EXTRACTED:    {ex['extracted']}")
    print()

### 6f. Low-accuracy banner

In [ ]:
if overall_acc < 0.30:
    banner = (
        '\n' + '!' * 78 + '\n'
        '!! VAL ACC LOW — review before submitting' + ' ' * 33 + '!!\n'
        '!! overall = ' + f'{overall_acc:.4f}' + ' (threshold 0.30)' + ' ' * 38 + '!!\n'
        '!! likely causes: under-trained adapter, broken chat template, format drift !!\n'
        '!! inspect the per-category table + debug printouts above before proceeding !!\n'
        + '!' * 78 + '\n'
    )
    print(banner)
else:
    print(f'val acc {overall_acc:.4f} >= 0.30 — proceeding to package submission')

## 7. Save adapter and package submission.zip

In [ ]:
model.save_pretrained(OUTPUT_DIR)
import os, subprocess
os.chdir(OUTPUT_DIR)
subprocess.run('zip -m submission.zip adapter_config.json adapter_model.safetensors', shell=True, check=True)
print('Wrote /kaggle/working/submission.zip')